In [44]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()


password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
db = os.getenv("DB_NAME")
user = os.getenv("DB_USER")


engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}/{db}")

In [45]:
query = "SELECT * FROM loan_data"
df = pd.read_sql(query, engine)
print(df.shape)

(150000, 12)


In [46]:
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [47]:
df = df.drop(columns=["Unnamed: 0"], axis=1)

In [48]:
df.shape

(150000, 11)

In [49]:
df.columns.tolist()

['SeriousDlqin2yrs',
 'RevolvingUtilizationOfUnsecuredLines',
 'age',
 'NumberOfTime30-59DaysPastDueNotWorse',
 'DebtRatio',
 'MonthlyIncome',
 'NumberOfOpenCreditLinesAndLoans',
 'NumberOfTimes90DaysLate',
 'NumberRealEstateLoansOrLines',
 'NumberOfTime60-89DaysPastDueNotWorse',
 'NumberOfDependents']

In [50]:
# ab hm outlier handle krenge
df = df[(df['age'] >= 18) & (df['age'] <= 95)]
df

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
149995,0,0.040674,74,0,0.225131,2100.0,4,0,1,0,0.0
149996,0,0.299745,44,0,0.716562,5584.0,4,0,1,0,2.0
149997,0,0.246044,58,0,3870.000000,NaN,18,0,1,0,0.0
149998,0,0.000000,30,0,0.000000,5716.0,4,0,0,0,0.0


In [51]:
df.shape

(149936, 11)

In [52]:
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=1.0)

In [53]:
print(df['RevolvingUtilizationOfUnsecuredLines'].min())
print(df['RevolvingUtilizationOfUnsecuredLines'].max())

0.0
1.0


In [54]:
df['DebtRatio'] = df['DebtRatio'].clip(upper=10)
print(df['DebtRatio'].min())
print(df['DebtRatio'].max())

0.0
10.0


In [55]:
df['NumberOfTime30-59DaysPastDueNotWorse'] = df['NumberOfTime30-59DaysPastDueNotWorse'].clip(upper=24)
df['NumberOfTime60-89DaysPastDueNotWorse'] = df['NumberOfTime60-89DaysPastDueNotWorse'].clip(upper=24)
df['NumberOfTimes90DaysLate'] = df['NumberOfTimes90DaysLate'].clip(upper=24)

In [56]:
print(df['NumberOfTime30-59DaysPastDueNotWorse'].max())
print(df['NumberOfTime60-89DaysPastDueNotWorse'].max())
print(df['NumberOfTimes90DaysLate'].max())

24
24
24


In [57]:
df['NumberOfOpenCreditLinesAndLoans'] = df['NumberOfOpenCreditLinesAndLoans'].clip(upper=30)

In [58]:
print(df['NumberOfOpenCreditLinesAndLoans'].max())

30


In [59]:
df['NumberRealEstateLoansOrLines'] = df['NumberRealEstateLoansOrLines'].clip(upper=10)

In [60]:
print(df['NumberRealEstateLoansOrLines'].max())

10


In [61]:
import numpy as np
print(np.__version__)

2.4.6


In [62]:
median_income = df['MonthlyIncome'].median()
print(median_income)

AttributeError: module 'numpy' has no attribute 'median'

In [ ]:
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(median_income)
print(df["MonthlyIncome"].isnull().sum())

0


In [ ]:
median_depend = df['NumberOfDependents'].median()
median_depend


np.float64(0.0)

In [ ]:
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(median_depend)
print(df['NumberOfDependents'].isnull().sum())

0


In [ ]:
print(df.isnull().sum())

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64


# Feature Engineering

In [ ]:
df['MonthlyDebtPayment'] = df['MonthlyIncome'] * df['DebtRatio']
print(df['MonthlyDebtPayment'])

0          7323.197016
1           316.878123
2           258.914887
3           118.963951
4          1584.975094
              ...     
149995      472.774869
149996     4001.283436
149997    54000.000000
149998        0.000000
149999     2038.750092
Name: MonthlyDebtPayment, Length: 149936, dtype: float64


In [ ]:
#df['TotalLate'] = df['NumberOfTime30-59DaysPastDueNotWorse'] + df['NumberOfTime60-89DaysPastDueNotWorse'] + df['NumberOfTimes90DaysLate']
#print(df['TotalLate'])